In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.special import gammaln

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "figure.dpi":         150,
})

TEAL   = ( 30/255, 130/255, 130/255)
SAND   = (184/255, 146/255,  58/255)
RED    = (177/255,  56/255,  69/255)

PANELS = [
    (0.25, r"$q\!=\!0.25$", TEAL),
    (0.50, r"$q\!=\!0.50$", SAND),
    (0.75, r"$q\!=\!0.75$", RED),
]

def _bar_color(rgb, x, x_max, pale_mix=0.72):
    t = min(abs(x) / x_max, 1.0)
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def _edge_color(rgb, x, x_max, pale_mix=0.35):
    t = min(abs(x) / x_max, 1.0)
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def simulate_erw_batch(n, p, q, n_sims, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.zeros((n_sims, n + 1), dtype=np.int8)
    xi[:, 1] = np.where(rng.random(n_sims) < q, 1, -1)
    for t in range(1, n):
        idx  = rng.integers(1, t + 1, size=n_sims)
        past = xi[np.arange(n_sims), idx]
        flip = np.where(rng.random(n_sims) < p, 1, -1)
        xi[:, t + 1] = past * flip
    return xi[:, 1:].sum(axis=1).astype(float)

N_STEPS = 5_000
N_SIMS  = 15_000
N_BINS  = 52
P       = 0.90
Q_VALUES = [0.25, 0.50, 0.75]

rng = np.random.default_rng(2026)

alpha = 2 * P - 1

scale = np.exp(alpha * np.log(N_STEPS) - gammaln(alpha + 1))

finals = {}
for q_val in Q_VALUES:
    beta = 2 * q_val - 1
    raw  = simulate_erw_batch(N_STEPS, p=P, q=q_val, n_sims=N_SIMS, rng=rng)
    finals[q_val] = raw / scale - beta

x_lim = 2.5
all_counts = []
bin_edges_common = np.linspace(-x_lim, x_lim, N_BINS + 1)

for q_val in Q_VALUES:
    c, _ = np.histogram(finals[q_val], bins=bin_edges_common, density=True)
    all_counts.append(c)

y_max = max(c.max() for c in all_counts) * 1.08

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.1))
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.19, top=0.80, wspace=0.28)

LABEL_GREY = (0.55, 0.55, 0.55)
bin_width = bin_edges_common[1] - bin_edges_common[0]
bin_centers = 0.5 * (bin_edges_common[:-1] + bin_edges_common[1:])

for ax, (q_val, label, color), counts in zip(axes, PANELS, all_counts):
    for xc, h in zip(bin_centers, counts):
        ax.bar(xc, h, width=bin_width,
               color=_bar_color(color, xc, x_lim),
               edgecolor=_edge_color(color, xc, x_lim),
               linewidth=0.28, alpha=0.70, zorder=2)

    ax.set_title(label, fontsize=9, pad=5, color=LABEL_GREY)
    ax.set_xlabel(r"$Z_n$", labelpad=3)
    ax.set_xlim(-x_lim, x_lim)
    ax.set_ylim(0, y_max)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator([-2, 0, 2]))
    ax.xaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, _:
            r"$-2$" if x == -2 else (r"$2$" if x == 2 else r"$0$"))
    )
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))

shared_yticks = axes[0].get_yticks()
for ax in axes[1:]:
    ax.set_yticks(shared_yticks)
    ax.set_ylim(0, y_max)

    ax.tick_params(axis="y", labelleft=False)

axes[0].set_ylabel("Density", labelpad=4, fontsize=8)

fig.text(0.5, 0.965, r"$p\!=\!0.90$", ha="center", va="top",
         fontsize=9, color=LABEL_GREY)

fig.savefig("figure_4_7.pdf", bbox_inches="tight", pad_inches=0.02)
print("Saved figure_4_7.pdf")